In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import numpy as np # linear algebra
import pandas as pd
import sqlite3

df_olist_customers = pd.read_csv('/content/drive/MyDrive/data/olist_customers_dataset.csv')
df_olist_sellers = pd.read_csv('/content/drive/MyDrive/data/olist_sellers_dataset.csv')
df_olist_order_reviews= pd.read_csv('/content/drive/MyDrive/data/olist_order_reviews_dataset.csv')
df_olist_order_items= pd.read_csv('/content/drive/MyDrive/data/olist_order_items_dataset.csv')
df_olist_products= pd.read_csv('/content/drive/MyDrive/data/olist_products_dataset.csv')
df_olist_geolocation= pd.read_csv('/content/drive/MyDrive/data/olist_geolocation_dataset.csv')
df_product_category_name_translation= pd.read_csv('/content/drive/MyDrive/data/product_category_name_translation.csv')
df_olist_orders = pd.read_csv('/content/drive/MyDrive/data/olist_orders_dataset.csv')
df_olist_order_payments= pd.read_csv('/content/drive/MyDrive/data/olist_order_payments_dataset.csv')

df_olist_customers.head()

from sqlalchemy import create_engine
engine = create_engine('sqlite://', echo=False)

# export the dataframe as a table 'playstore' to the sqlite engine
df_olist_customers.to_sql("olist_customers", con =engine)
df_olist_sellers.to_sql("olist_sellers", con =engine)
df_olist_order_reviews.to_sql("olist_order_reviews", con =engine)
df_olist_order_items.to_sql("olist_order_items", con =engine)
df_olist_products.to_sql("olist_products_dataset", con =engine)
df_olist_geolocation.to_sql("olist_geolocation", con =engine)
df_product_category_name_translation.to_sql("product_category_name_translation", con =engine)
df_olist_orders.to_sql("olist_orders", con =engine)
df_olist_order_payments.to_sql("olist_order_payments", con =engine)
df_olist_order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [ ]:
sql='''

Select * from olist_customers
limit 5


''';


df_sql = pd.read_sql_query(sql,con=engine)
df_sql.head()

In [ ]:
from sqlalchemy import inspect

insp = inspect(engine)
for table in insp.get_table_names():
  print(table, [col['name']for col in insp.get_columns(table)])

In [ ]:
sql = '''
SELECT COUNT(DISTINCT olist_orders.order_id) AS total_orders,
COUNT(DISTINCT CASE WHEN olist_order_reviews.review_score = 5 THEN olist_orders.order_id END) AS five_star_orders,
ROUND(COUNT(DISTINCT CASE WHEN olist_order_reviews.review_score = 5 THEN olist_orders.order_id END) * 100.0 / COUNT(DISTINCT olist_orders.order_id), 2) AS pct_five_star
FROM olist_orders
JOIN olist_order_reviews ON olist_orders.order_id = olist_order_reviews.order_id
WHERE olist_orders.order_purchase_timestamp LIKE '2018-01%'
'''

df_sql = pd.read_sql_query(sql, con=engine)
df_sql

In [ ]:
sql = '''
SELECT COUNT(order_purchase_timestamp) AS total_orders,
  strftime('%Y', order_purchase_timestamp) AS year,
  strftime('%m', order_purchase_timestamp) AS month
FROM olist_orders
GROUP BY year, month
ORDER BY year, month
'''
df_sql = pd.read_sql_query(sql, con=engine)
df_sql

In [ ]:
sql='''
SELECT customer_unique_id,
  COUNT(order_id) AS num_orders,
  ROUND(AVG(total_price), 2) AS avg_order_value
FROM (
  SELECT olist_order_items.order_id AS order_id,
    customer_unique_id,
    SUM(price) AS total_price
  FROM olist_order_items
  JOIN olist_orders ON olist_order_items.order_id = olist_orders.order_id
  JOIN olist_customers ON olist_orders.customer_id = olist_customers.customer_id
  GROUP BY olist_order_items.order_id
) AS order_totals
GROUP BY customer_unique_id
ORDER BY num_orders DESC
''';


df_sql = pd.read_sql_query(sql,con=engine)
df_sql.head()

In [15]:
sql = '''
SELECT customer_city,
    SUM(price) AS total_price
  FROM olist_order_items
  JOIN olist_orders ON olist_order_items.order_id = olist_orders.order_id
  JOIN olist_customers ON olist_orders.customer_id = olist_customers.customer_id
  WHERE strftime('%Y', order_purchase_timestamp) >= '2016' AND strftime('%Y', order_purchase_timestamp) <= '2018' AND order_status = 'delivered'
  GROUP BY customer_city
  ORDER BY total_price DESC
  LIMIT 5
'''
df_sql = pd.read_sql_query(sql, con=engine)
df_sql

,customer_city,total_price
0,sao paulo,1859556.83
1,rio de janeiro,955573.97
2,belo horizonte,346039.04
3,brasilia,295814.72
4,curitiba,205999.33


In [17]:
sql = '''
SELECT customer_state,
    SUM(price) AS total_price
  FROM olist_order_items
  JOIN olist_orders ON olist_order_items.order_id = olist_orders.order_id
  JOIN olist_customers ON olist_orders.customer_id = olist_customers.customer_id
  WHERE strftime('%Y', order_purchase_timestamp) >= '2016' AND strftime('%Y', order_purchase_timestamp) <= '2018' AND order_status = 'delivered'
  GROUP BY customer_state
  ORDER BY total_price DESC
'''
df_sql = pd.read_sql_query(sql, con=engine)
df_sql

,customer_state,total_price
0,SP,5067633.16
1,RJ,1759651.13
2,MG,1552481.83
3,RS,728897.47
4,PR,666063.51
5,SC,507012.13
6,BA,493584.14
7,DF,296498.41
8,GO,282836.70
9,ES,268643.45


In [34]:
sql = '''
SELECT seller_id,
  SUM(price) AS total_price,
  COUNT(*) AS goods_sold,
  COUNT(DISTINCT customer_unique_id) AS unique_customers,
  COUNT(DISTINCT five_star.order_id) AS five_star_orders
FROM olist_order_items
JOIN olist_orders ON olist_order_items.order_id = olist_orders.order_id
JOIN olist_customers ON olist_orders.customer_id = olist_customers.customer_id
LEFT JOIN (SELECT DISTINCT order_id
FROM olist_order_reviews WHERE review_score = 5 ) AS five_star ON olist_order_items.order_id = five_star.order_id
WHERE strftime('%Y', order_purchase_timestamp) >= '2016' AND strftime('%Y', order_purchase_timestamp) <= '2018' AND order_status = 'delivered' AND  order_status = 'delivered'
GROUP BY seller_id
ORDER BY total_price DESC
LIMIT 10
'''
df_sql = pd.read_sql_query(sql, con=engine)
df_sql

,seller_id,total_price,goods_sold,unique_customers,five_star_orders
0,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93,1148,1117,671
1,53243585a1d6dc2643021fd1853d8905,217940.44,400,340,199
2,4a3ca9315b744ce9f8e9374361493884,196882.12,1949,1759,859
3,fa1c13f2614d7b5c4749cbc52fecda94,190917.14,579,575,393
4,7c67e1448b00f6e969d365cea6b010ab,186570.05,1355,962,342
5,7e93a43ef30c4f03f38b393420bc753a,165981.49,322,319,211
6,da8622b14eb17ae2831f4ac5b9dab84a,159816.87,1548,1272,771
7,7a67c85e85bb2ce8582c35f2203ad736,139658.69,1155,1140,710
8,1025f0e2d44d7041d6cf58b6550e0bfa,138208.56,1420,894,502
9,955fee9216a65b617aa5c0531780ce60,131836.71,1472,1256,715


In [36]:
sql = '''
SELECT customer_state,
  COUNT(*) AS total_orders,
  SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
  ROUND(SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS delivery_rate
FROM olist_orders
JOIN olist_customers ON olist_orders.customer_id = olist_customers.customer_id
GROUP BY customer_state
ORDER BY delivery_rate DESC
'''
df_sql = pd.read_sql_query(sql, con=engine)
df_sql

,customer_state,total_orders,delivered_orders,delivery_rate
0,AC,81,80,98.77
1,AP,68,67,98.53
2,ES,2033,1995,98.13
3,MS,715,701,98.04
4,AM,148,145,97.97
5,TO,280,274,97.86
6,RS,5466,5345,97.79
7,RN,485,474,97.73
8,MT,907,886,97.68
9,PR,5045,4923,97.58


In [37]:
sql = '''
WITH counts AS (
  SELECT COALESCE(t.product_category_name_english, p.product_category_name) AS category,
         pay.payment_type,
         COUNT(DISTINCT i.order_id) AS num_orders
  FROM olist_order_items i
  JOIN olist_products_dataset p ON i.product_id = p.product_id
  LEFT JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
  JOIN olist_order_payments pay ON i.order_id = pay.order_id
  GROUP BY category, pay.payment_type
),
ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY category ORDER BY num_orders DESC) AS rn
  FROM counts
)
SELECT category, payment_type AS preferred_payment, num_orders
FROM ranked
WHERE rn = 1
ORDER BY num_orders DESC
'''
df_sql = pd.read_sql_query(sql, con=engine)
df_sql

,category,preferred_payment,num_orders
0,bed_bath_table,credit_card,7540
1,health_beauty,credit_card,6874
2,sports_leisure,credit_card,5904
3,furniture_decor,credit_card,4919
4,computers_accessories,credit_card,4737
...,...,...,...
69,cds_dvds_musicals,credit_card,9
70,portateis_cozinha_e_preparadores_de_alimentos,credit_card,8
71,fashion_childrens_clothes,credit_card,5
72,pc_gamer,credit_card,5


In [38]:
sql = '''
WITH city_coords AS (
  SELECT geolocation_city AS city, geolocation_state AS state,
         AVG(geolocation_lat) AS lat, AVG(geolocation_lng) AS lng
  FROM olist_geolocation
  GROUP BY geolocation_city, geolocation_state
),
pairs AS (
  SELECT DISTINCT c.customer_city, c.customer_state, s.seller_city, s.seller_state
  FROM olist_order_items i
  JOIN olist_orders o ON i.order_id = o.order_id
  JOIN olist_customers c ON o.customer_id = c.customer_id
  JOIN olist_sellers s ON i.seller_id = s.seller_id
)
SELECT p.customer_city, p.customer_state, p.seller_city, p.seller_state,
  ROUND(6371 * ACOS(MIN(1,
      COS(RADIANS(cc.lat)) * COS(RADIANS(sc.lat)) * COS(RADIANS(sc.lng) - RADIANS(cc.lng))
    + SIN(RADIANS(cc.lat)) * SIN(RADIANS(sc.lat)))), 1) AS distance_km
FROM pairs p
JOIN city_coords cc ON p.customer_city = cc.city AND p.customer_state = cc.state
JOIN city_coords sc ON p.seller_city = sc.city AND p.seller_state = sc.state
ORDER BY distance_km DESC
LIMIT 10
'''
df_sql = pd.read_sql_query(sql, con=engine)
df_sql

,customer_city,customer_state,seller_city,seller_state,distance_km
0,pau d'arco,AL,ibitinga,SP,9474.7
1,ilha dos valadares,PR,sao bento do sul,SC,8677.9
2,porto trombetas,PA,ribeirao preto,SP,5793.5
3,vila dos cabanos,PA,praia grande,SP,5344.9
4,vila dos cabanos,PA,santo andre,SP,5310.6
5,vila dos cabanos,PA,salto,SP,5296.5
6,ibiajara,BA,sao paulo,SP,4777.1
7,boa vista,RR,bombinhas,SC,3581.2
8,boa vista,RR,fazenda rio grande,PR,3396.2
9,boa vista,RR,sao jose dos pinhais,PR,3388.7
